In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [3]:
import os
from langgraph.checkpoint.postgres import PostgresSaver
from graph import build_graph

DB_URI = os.environ.get("POSTGRES_URI", "postgresql://localhost:5432/stardew_agent")

# .from_conn_string() 뒤에 꼭 .with_conn()을 붙여서 알맹이(Context)를 강제로 엽니다.
manager = PostgresSaver.from_conn_string(DB_URI)
checkpointer = manager.__enter__()  # 수동으로 상자 열기 (연결 유지)

checkpointer.setup()  # 최초 1회 테이블 생성
graph = build_graph(checkpointer)
print("그래프 준비 완료 및 DB 연결 유지 중")

그래프 준비 완료 및 DB 연결 유지 중


## 시나리오 1: thread_id 분리 검증

같은 `thread_id`로 두 번 호출하면 이전 대화를 기억하고,  
다른 `thread_id`로 호출하면 처음부터 시작하는지 확인.

In [4]:
from langchain_core.messages import HumanMessage

config_a = {"configurable": {"thread_id": "thread-1"}, "recursion_limit": 25}

result1 = graph.invoke(
    {"messages": [HumanMessage(content="내 이름은 민준이야. 기억해줘!")],
     "tool_call_trace": [], "skill_choices": []},
    config=config_a,
)
print("[1차 호출]")
print(result1["messages"][-1].content)

[1차 호출]
사용자의 이름은 민준입니다. 사용자가 처음에 자신의 이름을 민준이라고 알려주었고, 이후 이름을 물어보았기 때문에 사용자의 이름은 민준임을 확실히 알 수 있습니다.

---
📊 사용 도구: 사용자가 '내 이름은 민준이야. 기억해줘!'라고 말함, 사용자가 '내 이름이 뭐야?'라고 질문함
🧠 추론: 사용자가 자신의 이름을 민준이라고 명확히 밝혔고, 이후 이름을 물어보는 질문에 일관되게 민준이라고 답변하였으므로, 사용자의 이름은 민준임을 확신할 수 있습니다.
✅ 확신도: 100%


In [5]:
# 같은 thread_id — 이전 대화 이어받아야 함
result2 = graph.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭐야?")],
     "tool_call_trace": [], "skill_choices": []},
    config=config_a,
)
print("[같은 thread_id 2차 호출 — 이름 기억해야 함]")
print(result2["messages"][-1].content)

[같은 thread_id 2차 호출 — 이름 기억해야 함]
사용자의 이름은 민준입니다.

---
📊 사용 도구: 사용자가 '내 이름은 민준이야. 기억해줘!'라고 말함, 사용자가 '내 이름이 뭐야?'라고 질문함
🧠 추론: 사용자가 처음에 자신의 이름을 민준이라고 알려주었고, 이후 이름을 물어보았기 때문에 사용자의 이름은 민준임을 확실히 알 수 있습니다.
✅ 확신도: 100%


In [6]:
# 다른 thread_id — 새 세션, 이름 모름
config_b = {"configurable": {"thread_id": "thread-2"}, "recursion_limit": 25}
result3 = graph.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭐야?")],
     "tool_call_trace": [], "skill_choices": []},
    config=config_b,
)
print("[다른 thread_id — 이름 모름]")
print(result3["messages"][-1].content)

print("\n--- skill_choices (비어있어야 함) ---")
print(graph.get_state(config_b).values["skill_choices"])

[다른 thread_id — 이름 모름]
사용자의 이름은 대화 내역에 포함되어 있지 않아 알 수 없습니다.

---
📊 사용 도구: 대화 내역
🧠 추론: 사용자가 자신의 이름을 묻는 질문에 대해 이전 대화에서 이름이 언급된 적이 없으므로 알 수 없다는 점을 설명했습니다.
✅ 확신도: 100%

--- skill_choices (비어있어야 함) ---
[]


## 시나리오 2: interrupt 흐름

1. "농사 5렙인데 뭐 찍을지 고민이야" → agent가 선택지 안내 후 `[AWAITING_SKILL_CHOICE]`로 라우팅
2. `interrupt_before=["commit_choice"]`에 의해 그래프 일시정지
3. 사용자 답변을 state에 추가 → `invoke(None, config)`로 resume
4. `skill_choices`에 `{"level": 5, "choice": "목축업자"}` 저장 확인

In [7]:
config_skill = {"configurable": {"thread_id": "thread-skill-v2"}, "recursion_limit": 25}

result_q = graph.invoke(
    {"messages": [HumanMessage(content="농사 5렙인데 뭐 찍을지 고민이야")],
     "tool_call_trace": [], "skill_choices": []},
    config=config_skill,
)

print("[봇 응답 — 선택지 제시 후 멈춤]")
# 내부 시그널 [AWAITING_SKILL_CHOICE]가 포함돼 있음
print(result_q["messages"][-1].content)

[봇 응답 — 선택지 제시 후 멈춤]
농사 5레벨에서 선택할 수 있는 두 가지 직업은 목축업자와 경작인입니다.

- 목축업자: 동물 생산물의 가치가 20% 증가해요. 동물 키우는 걸 좋아하거나 우유, 계란 같은 생산물을 더 많이 팔고 싶다면 좋아요.
- 경작인: 작물의 가치가 10% 증가해요. 작물을 주로 재배하고 수익을 극대화하고 싶다면 좋은 선택이에요.

어떤 스타일이 더 마음에 드나요? 더 자세히 비교해 드릴 수도 있어요! [AWAITING_SKILL_CHOICE]


In [8]:
# 멈춘 위치 — ('commit_choice',) 이어야 함
snapshot = graph.get_state(config_skill)
print("다음 실행 노드:", snapshot.next)

다음 실행 노드: ('commit_choice',)


In [9]:
# 사용자 답변을 invoke에 직접 전달해 resume
# update_state(as_node 없음)는 pending task(commit_choice)가 이 dict를 반환한 것으로
# 처리해 commit_choice 함수를 건너뛰므로, invoke에 직접 넘긴다.
result_resume = graph.invoke(
    {"messages": [HumanMessage(content="목축업자 할래")]},
    config=config_skill,
)
print("[resume 후 봇 응답]")
print(result_resume["messages"][-1].content)

[resume 후 봇 응답]
목축업자를 선택하셨군요! 동물 생산물 가치가 20% 증가해서 우유, 계란 등 동물 관련 수익이 더 올라가니 동물 키우기에 큰 도움이 될 거예요. 앞으로 10레벨 때 닭장의달인과 양치기 중 선택도 기다리고 있으니 기대하세요! [AWAITING_SKILL_CHOICE]


In [10]:
state_s2 = graph.get_state(config_skill)
print("skill_choices:", state_s2.values["skill_choices"])
# 기대값: [{'level': 5, 'choice': '목축업자'}]

skill_choices: [{'level': 5, 'choice': '목축업자'}]


## 시나리오 3: 하수구 직업 변경 — 커스텀 reducer 검증

같은 thread에서 level 5 선택을 경작인으로 바꿀 때,  
`upsert_skill_choices`가 기존 항목을 교체하는지(리스트 길이 1 유지) 검증.

In [11]:
# 시나리오 2 thread 그대로 이어서 진행
result_change = graph.invoke(
    {"messages": [HumanMessage(content="사실 경작인으로 바꿨어")]},
    config=config_skill,
)
  
print("[봇 응답]")
print(result_change["messages"][-1].content)

[봇 응답]
경작인으로 변경하셨군요! 작물 가치가 10% 증가해서 농사 수익을 더 올릴 수 있어요. 10레벨 때는 장인(가공품 가치 +40%)과 농업전문가(곡물류 가치 +50%) 중에서 선택할 수 있으니, 작물 가공이나 곡물 재배 쪽으로도 전략을 세워보시면 좋겠네요. [AWAITING_SKILL_CHOICE]


In [12]:
# commit_choice 앞에서 다시 멈췄는지 확인 후 resume
snapshot3 = graph.get_state(config_skill)
print("멈춘 위치:", snapshot3.next)

# "사실 경작인으로 바꿨어"가 이미 메시지에 있으므로 바로 resume
result_resume3 = graph.invoke(None, config=config_skill)
print("[resume 후 봇 응답]")
print(result_resume3["messages"][-1].content)

멈춘 위치: ('commit_choice',)
[DEBUG commit_choice] level=5, choice=경작인
[resume 후 봇 응답]
사용자는 농사 5레벨에서 직업 선택을 고민하다가 목축업자와 경작인 두 가지 옵션을 고려했습니다. 처음에는 목축업자를 선택했으나, 이후 경작인으로 변경하였습니다. 목축업자는 동물 생산물 가치가 20% 증가하여 동물 관련 수익을 높이는 데 유리하고, 경작인은 작물 가치가 10% 증가하여 농작물 수익을 극대화할 수 있습니다. 10레벨 때는 각각 닭장의달인과 양치기, 장인과 농업전문가 중에서 추가 선택이 가능합니다. 사용자의 최종 선택은 경작인입니다.

---
📊 사용 도구: 대화 내용
🧠 추론: 대화에서 사용자가 농사 5레벨 직업 선택에 대해 고민하며 목축업자와 경작인 두 가지 옵션을 고려했고, 최종적으로 경작인을 선택한 점을 반영하여 두 직업의 특징과 사용자의 선택을 요약했습니다.
✅ 확신도: 100%


In [13]:
state_final = graph.get_state(config_skill)
choices = state_final.values["skill_choices"]
print("최종 skill_choices:", choices)

assert len(choices) == 1, f"upsert 실패 — 리스트 길이 {len(choices)}"
assert choices[0]["choice"] == "경작인", f"변경 실패 — 현재: {choices[0]['choice']}"
print("✅ upsert_skill_choices 검증 통과")

최종 skill_choices: [{'level': 5, 'choice': '경작인'}]
✅ upsert_skill_choices 검증 통과


In [14]:
state_debug = graph.get_state(config_skill)
print("전체 데이터:", state_debug.values)

전체 데이터: {'messages': [HumanMessage(content='농사 5렙인데 뭐 찍을지 고민이야', additional_kwargs={}, response_metadata={}, id='4ab47893-fd33-48da-b57f-076801fb5905'), AIMessage(content='농사 5레벨에서 선택할 수 있는 두 가지 직업은 목축업자와 경작인입니다.\n\n- 목축업자: 동물 생산물의 가치가 20% 증가해요. 동물 키우는 걸 좋아하거나 우유, 계란 같은 생산물을 더 많이 팔고 싶다면 좋아요.\n- 경작인: 작물의 가치가 10% 증가해요. 작물을 주로 재배하고 수익을 극대화하고 싶다면 좋은 선택이에요.\n\n어떤 스타일이 더 마음에 드나요? 더 자세히 비교해 드릴 수도 있어요! [AWAITING_SKILL_CHOICE]', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 782, 'total_tokens': 922, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c422928e22', 'id': 'chatcmpl-DiXALQlhHL8yP7dGOkgbJJtrsqAD4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id

## Postgres 체크포인트 적재 확인

In [15]:
import psycopg

with psycopg.connect(DB_URI) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM checkpoints;")
        print("저장된 체크포인트 수:", cur.fetchone()[0])

저장된 체크포인트 수: 191
